In [205]:
import pandas as pd
import re
from collections import Counter

In [206]:
df = pd.read_excel('./data/공고원본_POSTING_ID포함.xlsx')

print("행/열 크기:", df.shape)
print("컬럼 목록:")
print(df.columns.tolist())

행/열 크기: (884, 11)
컬럼 목록:
['POSTING_ID', 'COMPANY_NAME', 'POSTED_DATE', 'DEADLINE', 'REGION', 'EMPLOYMENT_TYPE', 'EDUCATION', 'JOB_POSITION', 'TECH_STACK', 'SALARY', 'POSTING_URL']


C:\Users\smhrd\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [207]:
TARGET_COL = "TECH_STACK"

print(df[[TARGET_COL]].head(10).to_string(index=False))

                                        TECH_STACK
       Android, JAVA, Python, MySQL, AI, 머신러닝, 딥러닝
                     HTML5, CSS, JAVA, JSP, Spring
                                              클라우드
CSS, Django, Docker, Spring Boot, Spring Framework
                                       SQL, Python
          CSS, HTML, JAVA, Javascript, Jquery, JSP
                                          빅데이터, AI
                                React, Javascript 
                            HTML5, CSS, Javascript
                                                AI


In [208]:
def split_basic(text):
    """
    기본 분리:
    - 쉼표 기준 분리
    - 앞뒤 공백 제거
    """
    if pd.isna(text):
        return []
    
    parts = re.split(r",", str(text))
    return [p.strip() for p in parts if p.strip()]

In [209]:
sample_text = "JAVA, JSP, Spring, JQuery, Ajax, DB, Query"
print(split_basic(sample_text))

['JAVA', 'JSP', 'Spring', 'JQuery', 'Ajax', 'DB', 'Query']


In [210]:
raw_tokens = []

for text in df[TARGET_COL].fillna(""):
    raw_tokens.extend(split_basic(text))

raw_counter = Counter(raw_tokens)

raw_freq_df = pd.DataFrame(raw_counter.items(), columns=["raw_stack", "count"])
raw_freq_df = raw_freq_df.sort_values(["count", "raw_stack"], ascending=[False, True]).reset_index(drop=True)

print("전체 토큰 수:", len(raw_tokens))
print("중복 제거한 원본 표현 수:", raw_freq_df.shape[0])
print(raw_freq_df.head(20).to_string(index=False))

전체 토큰 수: 3780
중복 제거한 원본 표현 수: 363
  raw_stack  count
       JAVA    300
     Python    259
 Javascript    171
      React    148
        CSS    138
      MySQL    125
       HTML    119
        AWS    116
        Git    113
        JSP    104
     Spring    104
        SQL    101
      Linux     94
    Node.js     91
     Oracle     88
        API     64
      Figma     63
 JavaScript     60
      HTML5     58
Spring Boot     53


In [211]:
# 기술스택 후보그룹 검토용, 빈도표를 보고 정규화 사전 만들기
normalize_map = {
    'JQUERY' : 'jQuery',
    'JQuery' : 'jQuery',
    'Jquery' : 'jQuery',
    'jQuery' : 'jQuery',
    'jQurery' : 'jQuery',

    'MYSQL' : 'MySQL', 
    'MySQL' : 'MySQL', 
    'MySQl' : 'MySQL',

    'GIT' : 'Git',
    'Git' : 'Git',
    'git' : 'Git',

    'Spring Boot' : 'Spring Boot', 
    'SpringBoot' : 'Spring Boot',
    'Springboot' : 'Spring Boot',
    'Sprimg Boot' : 'Spring Boot', 
    'SprimgBoot' : 'Spring Boot',

    'TensorFlow' : 'TensorFlow', 
    'Tensorflow' : 'TensorFlow',
    'tensorflow' : 'TensorFlow',
    'Tensorflow)활용 경험' : 'TensorFlow',

    # React.js, React를 하나로 모음
    'React.JS' : 'React',
    'React.js' : 'React',
    'ReactJS' : 'React',
    'React' : 'React',
    'react' : 'React',

    'React Native' : 'React Native',
    'React native' : 'React Native',
    'React-Native' : 'React Native',

    # RESTful API, REST API를 하나로 모음
    'RESTful API' : 'REST API', 
    'Restful API' : 'REST API',
    'Restful APi' : 'REST API',
    'Resstful API' : 'REST API',
    'REST API' : 'REST API',
    'RestAPI' : 'REST API',

    'JAVA' : 'Java', 
    'Java' : 'Java',

    'JavaScript' : 'JavaScript',
    'Javascript' : 'JavaScript',
    'x-javascript' : 'JavaScript',
    'JavaScrpit' : 'JavaScript',

    'JSP' : 'JSP',
    'Jsp' : 'JSP',

    'Linux' : 'Linux',
    'Linux.' : 'Linux',
    'Linu' : 'Linux',

    'Node.js' : 'Node.js', 
    'Nodejs' : 'Node.js',

    'Ai' : 'AI',
    'AI' : 'AI',
    '인공지능(AI)' : 'AI',

    'PyTorch' : 'PyTorch',
    'Pytorch' : 'PyTorch',

    'AJAX' : 'Ajax',
    'Ajax' : 'Ajax',

    # CSS3를 CSS로 모음
    'CSS 3' : 'CSS', 
    'CSS3' : 'CSS',

    'Flutter' : 'Flutter', 
    'flutter' : 'Flutter',

    'GitHub' : 'GitHub', 
    'Github' : 'GitHub',

    'PHP' : 'PHP', 
    'php' : 'PHP',

    'BootStrap' : 'Bootstrap', 
    'Bootstrap' : 'Bootstrap',

    'Open CV' : 'OpenCV', 
    'OpenCV' : 'OpenCV',

    'TypeScript' : 'TypeScript', 
    'Typescript' : 'TypeScript',

    'MyBatis' : 'MyBatis',
    'mybatis' : 'MyBatis',

    'IBatis' : 'IBatis',
    'iBATIS' : 'IBatis',

    'Nest.js' : 'NestJS',
    'NestJS' : 'NestJS',

    'PostgreSQL' : 'PostgreSQL', 
    'postgresql' : 'PostgreSQL',
    'Postgrsql' : 'PostgreSQL',

    'Kubernetes' : 'Kubernetes', 
    'kubernetes' : 'Kubernetes',

    'Network' : 'Network', 
    'network' : 'Network',

    'iOS' : 'iOS', 
    'ios' : 'iOS',

    'Hugging Face' : 'Hugging Face',
    'HuggingFace' : 'Hugging Face',

    'Scikit-Learn' : 'Scikit-Learn', 
    'Scikit-learn' : 'Scikit-Learn',

    'TailwindCSS' : 'TailwindCSS', 
    'tailwindcss' : 'TailwindCSS',

    'WebSocket' : 'WebSocket', 
    'Websocket' : 'WebSocket',
    'WebSocke' : 'WebSocket',

    'Spring Framework' : 'Spring Framework',
    'Sping Framework' : 'Spring Framework',

    'Python' : 'Python',
    'Phython' : 'Python',
    'Pythoin' : 'Python',

    '전자정부프레임워크' : '전자정부표준프레임워크',
    '전자정부표준프레임워크' : '전자정부표준프레임워크',

    'Transformwe' : 'Transformer',
    'Transformer' : 'Transformer',

    'HTML5' : 'HTML',
    'HTML' : 'HTML',

    'Word' : '워드',
    '워드' : '워드',

    'C언어' : 'C',
    'C' : 'C',

    '.Net Framrwork' : '.NET Framework',
    'Diango' : 'Django',
    'Excle' : 'Excel',
    'HTLM' : 'HTML',
    'Seienium' : 'Selenium',
    'Spting' : 'Spring',
    'Uneral' : 'Unreal',
    'VScode' : 'VSCode',
    'swagger' : 'Swagger',
}

# 2) 분리 규칙
split_map = {
    'C++. Python': ['C++', 'Python'],
    'CISCO. Docker': ['CISCO', 'Docker'],
    'Git. Java': ['Git', 'Java'],
    'HTML/CSS': ['HTML', 'CSS'],
    'JavaScript/TypeScript': ['JavaScript', 'TypeScript'],
    'GO Java': ['Go', 'Java'],
    'Git MongoDB': ['Git', 'MongoDB'],
    'PHP RestAPY': ['PHP', 'REST API'],
    'PHP CI' : ['PHP', 'CodeIgniter'],
}

# 3) 제외 목록
exclude_tokens = {
    '2종보통운전면허',
    'ADsP 자격',
    '관련학과',
    '영어',
    '앱 개발',
    '웹개발',
    '웹 서비스',
    '프레젠테이션',
}

# 4) 보류 목록
hold_tokens = {
    'Query',
    'IT',
    '인공지능',
    '빅데이터',
}

In [212]:
normalize_map.update({
    'Google sheet': 'Google Sheets',
    '라즈베리파이': 'Raspberry Pi',
    '아두이노': 'Arduino',
    '어셈블리': 'Assembly',
    '임베디드리눅스': 'Embedded Linux',
    '딥러닝 프레임워크(Pytorch': 'PyTorch',
})

In [213]:
split_map.update({
    'Springm Git': ['Spring', 'Git'],
})

In [214]:
exclude_tokens.update({
    '웹디자인기능사 자격',
    'SQLD 자격',
    '논문 구현 및 모델 경량화 경험(Quantization',
    '마크업',
    '백앤드개발',
    '전자장부',
    '컴퓨터 공학과 및 관련학과 졸업',
    '패턴인식',
})

In [215]:
hold_tokens.update({
    'AL/ML 도메인',
    'Google Optimize',
    'SQLD',
    '관계형 데이터베이스',
    '데이터마이닝',
    '웹접근성',
    '플래시',
})

In [216]:
hold_tokens.discard('SQLD')
exclude_tokens.add('SQLD')

In [217]:
test_tokens = [
    '웹디자인기능사 자격',
    'AL/ML 도메인',
    'Google sheet',
    'Google Optimize',
    'SQLD',
    'SQLD 자격',
    'Springm Git',
    '관계형 데이터베이스',
    '논문 구현 및 모델 경량화 경험(Quantization',
    '데이터마이닝',
    '딥러닝 프레임워크(Pytorch',
    '라즈베리파이',
    '마크업',
    '백앤드개발',
    '아두이노',
    '어셈블리',
    '웹접근성',
    '임베디드리눅스',
    '전자장부',
    '컴퓨터 공학과 및 관련학과 졸업',
    '패턴인식',
    '플래시',
]

for token in test_tokens:
    n, h, e = process_token(token, normalize_map, split_map, exclude_tokens, hold_tokens)
    print(f"[원본] {token}")
    print("  normalized:", n)
    print("  hold      :", h)
    print("  excluded  :", e)
    print("-" * 50)

[원본] 웹디자인기능사 자격
  normalized: []
  hold      : []
  excluded  : ['웹디자인기능사 자격']
--------------------------------------------------
[원본] AL/ML 도메인
  normalized: []
  hold      : ['AL/ML 도메인']
  excluded  : []
--------------------------------------------------
[원본] Google sheet
  normalized: ['Google Sheets']
  hold      : []
  excluded  : []
--------------------------------------------------
[원본] Google Optimize
  normalized: []
  hold      : ['Google Optimize']
  excluded  : []
--------------------------------------------------
[원본] SQLD
  normalized: []
  hold      : []
  excluded  : ['SQLD']
--------------------------------------------------
[원본] SQLD 자격
  normalized: []
  hold      : []
  excluded  : ['SQLD 자격']
--------------------------------------------------
[원본] Springm Git
  normalized: ['Spring', 'Git']
  hold      : []
  excluded  : []
--------------------------------------------------
[원본] 관계형 데이터베이스
  normalized: []
  hold      : ['관계형 데이터베이스']
  excluded  : []
------------

In [218]:
normalize_map.update({
    '.Net': '.NET',
    'three.js': 'Three.js',
    'DeepLearning': 'Deep Learning',
    'Fullter': 'Flutter',
    'Moderm c++': 'Modern C++',
    'VIsualC.C++': 'Visual C++',
    'Vbscript': 'VBScript',
    'MS office': 'MS Office',
    'codelgniter': 'CodeIgniter',
    '리눅스': 'Linux',
    '와이어프레임': 'Wireframe',
    'NHN클라우드': 'NHN Cloud',
    'Vue': 'Vue.js',
    'Next': 'Next.js',
})

In [219]:
split_map.update({
    'API Git': ['API', 'Git'],
    'FastAPI(Python)': ['FastAPI', 'Python'],
    'Native AOS/IOS': ['AOS', 'iOS'],
    'React-Query SWR': ['React Query', 'SWR'],
    'RNN LSTM': ['RNN', 'LSTM'],
})

In [220]:
hold_tokens.update({
    'Agent',
    'Boot',
    'DL',
    'Flow',
    'JS',
    'MIL',
    'OA',
    'OpenAI GPT',
    'GPT API',
    'SQL Query',
    'Temporal',
    'Text Mining',
    'Torch',
    'Vision',
    'WEB',
    'UI/UX',
})

In [221]:
exclude_tokens.update({
    'PPT',
    'Wireframe',
    '와이어프레임',
    'Database ORM',
    'Relational Database',
    'SPA 프레임워크',
    'SEO',
})

In [222]:
normalize_map.update({
    'OracleDB': 'Oracle',

    'Cloud': '클라우드',
    '클라우드': '클라우드',

    'Excel': 'Microsoft Excel',
    'PowerPoint': 'Microsoft PowerPoint',
    '워드': 'Microsoft Word',
    'MS Office': 'Microsoft Office',
    'Microsoft Office': 'Microsoft Office',

    'Deep Learning': '딥러닝',
    '딥러닝': '딥러닝',

    'MachineLearning': '머신러닝',
    '머신러닝': '머신러닝',
    'ML': '머신러닝',

    'OpenAI GPT': 'GPT',
    'OpenAI': 'LLM API',

    'UI': 'UI/UX',
    'UX': 'UI/UX',

    'Pruning)': 'Pruning',
    'sLM': 'SLM',
    'gis tool': 'GIS Tool',
    'OpenSource': 'Open Source',
    'cursor': 'Cursor',
    'xPlatform': 'XPlatform',
})

In [223]:
split_map.update({
    'Linus HTML': ['Linux', 'HTML'],
    'C/C++': ['C', 'C++'],
})

In [224]:
normalize_map

{'JQUERY': 'jQuery',
 'JQuery': 'jQuery',
 'Jquery': 'jQuery',
 'jQuery': 'jQuery',
 'jQurery': 'jQuery',
 'MYSQL': 'MySQL',
 'MySQL': 'MySQL',
 'MySQl': 'MySQL',
 'GIT': 'Git',
 'Git': 'Git',
 'git': 'Git',
 'Spring Boot': 'Spring Boot',
 'SpringBoot': 'Spring Boot',
 'Springboot': 'Spring Boot',
 'Sprimg Boot': 'Spring Boot',
 'SprimgBoot': 'Spring Boot',
 'TensorFlow': 'TensorFlow',
 'Tensorflow': 'TensorFlow',
 'tensorflow': 'TensorFlow',
 'Tensorflow)활용 경험': 'TensorFlow',
 'React.JS': 'React',
 'React.js': 'React',
 'ReactJS': 'React',
 'React': 'React',
 'react': 'React',
 'React Native': 'React Native',
 'React native': 'React Native',
 'React-Native': 'React Native',
 'RESTful API': 'REST API',
 'Restful API': 'REST API',
 'Restful APi': 'REST API',
 'Resstful API': 'REST API',
 'REST API': 'REST API',
 'RestAPI': 'REST API',
 'JAVA': 'Java',
 'Java': 'Java',
 'JavaScript': 'JavaScript',
 'Javascript': 'JavaScript',
 'x-javascript': 'JavaScript',
 'JavaScrpit': 'JavaScript',
 '

In [225]:
for k, v in normalize_map.items():
    if v in normalize_map and normalize_map[v] != v:
        print(f"{k} -> {v} -> {normalize_map[v]}")

Word -> 워드 -> Microsoft Word
Excle -> Excel -> Microsoft Excel
DeepLearning -> Deep Learning -> 딥러닝
MS office -> MS Office -> Microsoft Office


In [226]:
normalize_map.update({
    'Word': 'Microsoft Word',
    '워드': 'Microsoft Word',
    'Excle' : 'Microsoft Excel',
    'Excel' : 'Microsoft Excel',
    'DeepLearning' : '딥러닝',
    'Deep Learning' : '딥러닝',
    'MS office' : 'Microsoft Office',
    'MS Office' : 'Microsoft Office'
})

In [227]:
print("normalize_map 개수:", len(normalize_map))
print("split_map 개수:", len(split_map))
print("exclude_tokens 개수:", len(exclude_tokens))
print("hold_tokens 개수:", len(hold_tokens))

normalize_map 개수: 158
split_map 개수: 17
exclude_tokens 개수: 24
hold_tokens 개수: 26


In [228]:
def unique_keep_order(items):
    """순서를 유지하면서 중복 제거"""
    return list(dict.fromkeys(items))

def process_token(token, normalize_map, split_map, exclude_tokens, hold_tokens):
    """
    토큰 1개 처리
    반환값:
    - normalized: 정규화 결과 리스트
    - held: 보류 리스트
    - excluded: 제외 리스트
    """
    normalized = []
    held = []
    excluded = []

    token = str(token).strip()

    if not token:
        return normalized, held, excluded

    # 1. 분리 규칙 우선 적용
    if token in split_map:
        for sub_token in split_map[token]:
            n, h, e = process_token(
                sub_token,
                normalize_map,
                split_map,
                exclude_tokens,
                hold_tokens
            )
            normalized.extend(n)
            held.extend(h)
            excluded.extend(e)
        return normalized, held, excluded

    # 2. 제외
    if token in exclude_tokens:
        excluded.append(token)
        return normalized, held, excluded

    # 3. 보류
    if token in hold_tokens:
        held.append(token)
        return normalized, held, excluded

    # 4. 정규화
    normalized_token = normalize_map.get(token, token)
    normalized.append(normalized_token)

    return normalized, held, excluded

def process_stack_text(text, normalize_map, split_map, exclude_tokens, hold_tokens):
    """
    TECH_STACK 문자열 전체 처리
    """
    tokens = split_basic(text)

    normalized_result = []
    held_result = []
    excluded_result = []

    for token in tokens:
        n, h, e = process_token(
            token,
            normalize_map,
            split_map,
            exclude_tokens,
            hold_tokens
        )
        normalized_result.extend(n)
        held_result.extend(h)
        excluded_result.extend(e)

    normalized_result = unique_keep_order(normalized_result)
    held_result = unique_keep_order(held_result)
    excluded_result = unique_keep_order(excluded_result)

    return normalized_result, held_result, excluded_result

In [229]:
test_tokens = [
    "JQuery",
    "Phython",
    "HTML/CSS",
    "Query",
]

for token in test_tokens:
    n, h, e = process_token(token, normalize_map, split_map, exclude_tokens, hold_tokens)
    print(f"[원본] {token}")
    print("  normalized:", n)
    print("  hold      :", h)
    print("  excluded  :", e)
    print("-" * 40)

[원본] JQuery
  normalized: ['jQuery']
  hold      : []
  excluded  : []
----------------------------------------
[원본] Phython
  normalized: ['Python']
  hold      : []
  excluded  : []
----------------------------------------
[원본] HTML/CSS
  normalized: ['HTML', 'CSS']
  hold      : []
  excluded  : []
----------------------------------------
[원본] Query
  normalized: []
  hold      : ['Query']
  excluded  : []
----------------------------------------


In [230]:
sample_text = "JAVA, JSP, Spring, JQuery, Ajax, DB, Query"
n, h, e = process_stack_text(sample_text, normalize_map, split_map, exclude_tokens, hold_tokens)

print("원본:", sample_text)
print("normalized:", n)
print("hold:", h)
print("excluded:", e)

원본: JAVA, JSP, Spring, JQuery, Ajax, DB, Query
normalized: ['Java', 'JSP', 'Spring', 'jQuery', 'Ajax', 'DB']
hold: ['Query']
excluded: []


In [231]:
normalized_col = []
held_col = []
excluded_col = []

for text in df[TARGET_COL].fillna(""):
    normalized_list, held_list, excluded_list = process_stack_text(
        text,
        normalize_map,
        split_map,
        exclude_tokens,
        hold_tokens
    )

    normalized_col.append(normalized_list)
    held_col.append(held_list)
    excluded_col.append(excluded_list)

df["TECH_STACK_NORMALIZED_LIST"] = normalized_col
df["TECH_STACK_HOLD_LIST"] = held_col
df["TECH_STACK_EXCLUDED_LIST"] = excluded_col

df["TECH_STACK_NORMALIZED"] = df["TECH_STACK_NORMALIZED_LIST"].apply(lambda x: ", ".join(x))
df["TECH_STACK_HOLD"] = df["TECH_STACK_HOLD_LIST"].apply(lambda x: ", ".join(x))
df["TECH_STACK_EXCLUDED"] = df["TECH_STACK_EXCLUDED_LIST"].apply(lambda x: ", ".join(x))

print("처리 완료")

처리 완료


In [232]:
sample_df = df[
    [
        "POSTING_ID",
        TARGET_COL,
        "TECH_STACK_NORMALIZED",
        "TECH_STACK_HOLD",
        "TECH_STACK_EXCLUDED"
    ]
].head(20).copy()

sample_df

,POSTING_ID,TECH_STACK,TECH_STACK_NORMALIZED,TECH_STACK_HOLD,TECH_STACK_EXCLUDED
0,1,"Android, JAVA, Python, MySQL, AI, 머신러닝, 딥러닝","Android, Java, Python, MySQL, AI, 머신러닝, 딥러닝",,
1,2,"HTML5, CSS, JAVA, JSP, Spring","HTML, CSS, Java, JSP, Spring",,
2,3,클라우드,클라우드,,
3,4,"CSS, Django, Docker, Spring Boot, Spring Framework","CSS, Django, Docker, Spring Boot, Spring Framework",,
4,5,"SQL, Python","SQL, Python",,
5,6,"CSS, HTML, JAVA, Javascript, Jquery, JSP","CSS, HTML, Java, JavaScript, jQuery, JSP",,
6,7,"빅데이터, AI",AI,빅데이터,
7,8,"React, Javascript","React, JavaScript",,
8,9,"HTML5, CSS, Javascript","HTML, CSS, JavaScript",,
9,10,AI,AI,,


In [233]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

def pretty_stack(text):
    if pd.isna(text) or str(text).strip() == "":
        return "-"
    items = [x.strip() for x in str(text).split(",") if x.strip()]
    return "\n  - " + "\n  - ".join(items)

def show_rows_vertical(dataframe, cols, n=10):
    """
    데이터프레임을 행별로 세로 출력
    """
    if dataframe.empty:
        print("표시할 행이 없음")
        return
    
    for idx, (_, row) in enumerate(dataframe.head(n).iterrows(), start=1):
        print(f"\n===== sample {idx} =====")
        for col in cols:
            print(f"[{col}]")
            print(pretty_stack(row[col]))
        print("-" * 50)

In [234]:
keywords = ["JQuery", "Phython", "Query", "HTML/CSS", "Sprimg", "React.js"]

for kw in keywords:
    temp = df[df[TARGET_COL].fillna("").str.contains(kw, na=False, regex=False)].copy()

    print(f"\n===== keyword: {kw} / 행 수: {len(temp)} =====")
    
    display(
        temp[
            [
                TARGET_COL,
                "TECH_STACK_NORMALIZED",
                "TECH_STACK_HOLD",
                "TECH_STACK_EXCLUDED"
            ]
        ].head(10)
    )


===== keyword: JQuery / 행 수: 4 =====


,TECH_STACK,TECH_STACK_NORMALIZED,TECH_STACK_HOLD,TECH_STACK_EXCLUDED
183,"Javascript, JQuery, Ajax, Bootstrap","JavaScript, jQuery, Ajax, Bootstrap",,
196,"HTML, CSS3, Javascript, JQuery","HTML, CSS, JavaScript, jQuery",,
370,"JAVA, JSP, Spring, RDBMS, JQuery, CSS, HTML5","Java, JSP, Spring, RDBMS, jQuery, CSS, HTML",,
376,"JAVA , Javascript, JQuery","Java, JavaScript, jQuery",,



===== keyword: Phython / 행 수: 1 =====


,TECH_STACK,TECH_STACK_NORMALIZED,TECH_STACK_HOLD,TECH_STACK_EXCLUDED
767,"Linux., Phython","Linux, Python",,



===== keyword: Query / 행 수: 27 =====


...

,TECH_STACK,TECH_STACK_NORMALIZED,TECH_STACK_HOLD,TECH_STACK_EXCLUDED
735,"React, Typescript, Next.js, Redux, Recoil, React-Query SWR, HTML/CSS","React, TypeScript, Next.js, Redux, Recoil, React Query, SWR, HTML, CSS",,



===== keyword: Sprimg / 행 수: 2 =====


,TECH_STACK,TECH_STACK_NORMALIZED,TECH_STACK_HOLD,TECH_STACK_EXCLUDED
411,"JAVA, JavaScript, SprimgBoot","Java, JavaScript, Spring Boot",,
628,"React.js, JavaScript, HTML, CSS, JAVA, Sprimg Boot, SQL, Oracle, MySQL","React, JavaScript, HTML, CSS, Java, Spring Boot, SQL, Oracle, MySQL",,



===== keyword: React.js / 행 수: 13 =====


,TECH_STACK,TECH_STACK_NORMALIZED,TECH_STACK_HOLD,TECH_STACK_EXCLUDED
38,"React.js, Node.js, AWS","React, Node.js, AWS",,
210,"JAVA, JSP, Spring, MyBatis, React.js, SQL, Git, Javascript, HTML5, CSS3, Oracle, MySQL","Java, JSP, Spring, MyBatis, React, SQL, Git, JavaScript, HTML, CSS, Oracle, MySQL",,
295,"API, Linux, React.js, Github, Notion, Node.js, AWS","API, Linux, React, GitHub, Notion, Node.js, AWS",,
313,"API, Linux, React.js, Github, Notion, Node.js, AWS","API, Linux, React, GitHub, Notion, Node.js, AWS",,
328,"JavaScript, React.js, AWS, HTML5, CSS3","JavaScript, React, AWS, HTML, CSS",,
439,"React.js, HTML, CSS, Javascript","React, HTML, CSS, JavaScript",,
448,"SpringBoot, React.js, JAVA, JSP, SQL, Figma, Git","Spring Boot, React, Java, JSP, SQL, Figma, Git",,
473,"Spring, Spring Boot, JAVA, React.js","Spring, Spring Boot, Java, React",,
522,"React.js, JavaScript, HTML, CSS, JAVA, Spring Boot, SQL, Oracle, MySQL, Git","React, JavaScript, HTML, CSS, Java, Spring Boot, SQL, Oracle, MySQL, Git",,
571,"HTML5, Javascript, React.js, AWS","HTML, JavaScript, React, AWS",,


In [235]:
df["HAS_HOLD"] = df["TECH_STACK_HOLD_LIST"].apply(lambda x: len(x) > 0)
df["HAS_EXCLUDED"] = df["TECH_STACK_EXCLUDED_LIST"].apply(lambda x: len(x) > 0)

review_df = df[df["HAS_HOLD"] | df["HAS_EXCLUDED"]].copy()

print("보류 또는 제외가 있는 행 수:", len(review_df))
display(
    review_df[
        [
            "POSTING_ID",
            TARGET_COL,
            "TECH_STACK_NORMALIZED",
            "TECH_STACK_HOLD",
            "TECH_STACK_EXCLUDED"
        ]
    ].head(20)
)

보류 또는 제외가 있는 행 수: 51


,POSTING_ID,TECH_STACK,TECH_STACK_NORMALIZED,TECH_STACK_HOLD,TECH_STACK_EXCLUDED
6,7,"빅데이터, AI",AI,빅데이터,
20,21,"JAVA, Javascript, JSP, MSSQL, MySQL, LLM, OpenAI GPT, Llama, Python","Java, JavaScript, JSP, MSSQL, MySQL, LLM, Llama, Python",OpenAI GPT,
29,30,"Python, Torch, HuggingFace","Python, Hugging Face",Torch,
35,36,"ADsP 자격, SQLD 자격",,,"ADsP 자격, SQLD 자격"
64,65,UI/UX,,UI/UX,
91,92,앱 개발,,,앱 개발
142,143,"JAVA, Javascript, JSP, MySQL, LLM, OpenAI GPT","Java, JavaScript, JSP, MySQL, LLM",OpenAI GPT,
325,326,"JAVA, MySQL, Oracle, Python, Linux, Ai, 인공지능, Spring, Spring Boot","Java, MySQL, Oracle, Python, Linux, AI, Spring, Spring Boot",인공지능,
329,330,"Jira, Excel, Figma, Google Optimize, Notion, PowerPoint, Asana","Jira, Microsoft Excel, Figma, Notion, Microsoft PowerPoint, Asana",Google Optimize,
422,423,"Notion, GPT API, AI","Notion, AI",GPT API,


In [236]:
def make_freq_df(list_series, col_name):
    all_items = []
    for row in list_series:
        all_items.extend(row)

    if not all_items:
        return pd.DataFrame(columns=[col_name, "count"])

    counter = Counter(all_items)
    freq_df = pd.DataFrame(counter.items(), columns=[col_name, "count"])
    freq_df = freq_df.sort_values(["count", col_name], ascending=[False, True]).reset_index(drop=True)
    return freq_df

In [237]:
normalized_freq_df = make_freq_df(df["TECH_STACK_NORMALIZED_LIST"], "tech_stack")
hold_freq_df = make_freq_df(df["TECH_STACK_HOLD_LIST"], "hold_token")
excluded_freq_df = make_freq_df(df["TECH_STACK_EXCLUDED_LIST"], "excluded_token")

print("정규화 후 기술스택 종류 수:", len(normalized_freq_df))
print("보류 토큰 종류 수:", len(hold_freq_df))
print("제외 토큰 종류 수:", len(excluded_freq_df))

정규화 후 기술스택 종류 수: 213
보류 토큰 종류 수: 26
제외 토큰 종류 수: 24


In [238]:
print(normalized_freq_df.head(50).to_string(index=False))

          tech_stack  count
                Java    331
              Python    263
          JavaScript    234
               React    173
                HTML    172
                 CSS    155
               MySQL    127
                 Git    120
                 AWS    116
              Oracle    109
                 JSP    105
              Spring    105
                 SQL    101
               Linux     99
             Node.js     92
         Spring Boot     72
                 API     65
               Figma     63
              jQuery     57
                  AI     52
          TensorFlow     51
             PyTorch     49
                머신러닝     40
                 딥러닝     37
                  C#     36
                 C++     36
             Android     33
     Microsoft Excel     27
              Notion     27
Microsoft PowerPoint     26
                DBMS     24
               RDBMS     23
                Ajax     20
             Flutter     20
                 PHP

In [239]:
print(hold_freq_df.to_string(index=False))

     hold_token  count
          Agent      2
             OA      2
     OpenAI GPT      2
          UI/UX      2
            WEB      2
      AL/ML 도메인      1
           Boot      1
             DL      1
           Flow      1
        GPT API      1
Google Optimize      1
             IT      1
             JS      1
            MIL      1
          Query      1
      SQL Query      1
       Temporal      1
    Text Mining      1
          Torch      1
         Vision      1
     관계형 데이터베이스      1
         데이터마이닝      1
           빅데이터      1
           웹접근성      1
           인공지능      1
            플래시      1


In [240]:
print(excluded_freq_df.to_string(index=False))

                excluded_token  count
                        와이어프레임      2
                    웹디자인기능사 자격      2
                      2종보통운전면허      1
                       ADsP 자격      1
                  Database ORM      1
                           PPT      1
           Relational Database      1
                           SEO      1
                     SPA 프레임워크      1
                          SQLD      1
                       SQLD 자격      1
                     Wireframe      1
                          관련학과      1
논문 구현 및 모델 경량화 경험(Quantization      1
                           마크업      1
                         백앤드개발      1
                          앱 개발      1
                            영어      1
                         웹 서비스      1
                           웹개발      1
                          전자장부      1
             컴퓨터 공학과 및 관련학과 졸업      1
                          패턴인식      1
                        프레젠테이션      1


In [241]:
print("원본 기술스택 종류 수:", raw_freq_df.shape[0])
print("정규화 후 기술스택 종류 수:", normalized_freq_df.shape[0])

원본 기술스택 종류 수: 363
정규화 후 기술스택 종류 수: 213


In [242]:
raw_stack_set = set(raw_freq_df["raw_stack"].tolist())
unused_keys = [k for k in normalize_map.keys() if k not in raw_stack_set]

print("사전에 넣었지만 원본 데이터에는 없는 키 수:", len(unused_keys))
print(unused_keys[:100])

사전에 넣었지만 원본 데이터에는 없는 키 수: 2
['MS Office', 'Deep Learning']


In [243]:
output_path = "공고원본_POSTING_ID포함_정리결과.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="job_posting_processed", index=False)
    review_df.to_excel(writer, sheet_name="review_rows", index=False)
    raw_freq_df.to_excel(writer, sheet_name="raw_freq", index=False)
    normalized_freq_df.to_excel(writer, sheet_name="normalized_freq", index=False)
    hold_freq_df.to_excel(writer, sheet_name="hold_freq", index=False)
    excluded_freq_df.to_excel(writer, sheet_name="excluded_freq", index=False)

print(f"저장 완료: {output_path}")

저장 완료: 공고원본_POSTING_ID포함_정리결과.xlsx


In [244]:
print("=== 최종 요약 ===")
print("전체 행 수:", len(df))
print("원본 기술스택 종류 수:", raw_freq_df.shape[0])
print("정규화 후 기술스택 종류 수:", normalized_freq_df.shape[0])
print("보류 포함 행 수:", df["HAS_HOLD"].sum())
print("제외 포함 행 수:", df["HAS_EXCLUDED"].sum())

=== 최종 요약 ===
전체 행 수: 884
원본 기술스택 종류 수: 363
정규화 후 기술스택 종류 수: 213
보류 포함 행 수: 30
제외 포함 행 수: 24


In [245]:
# Oracle 연결 후 실행
not_found_ids = []

for pid in update_df["POSTING_ID"]:
    cur.execute("""
        SELECT COUNT(*)
        FROM JOB_POSTING
        WHERE POSTING_ID = :pid
    """, {"pid": int(pid)})
    
    cnt = cur.fetchone()[0]
    if cnt == 0:
        not_found_ids.append(pid)

print("DB에서 못 찾은 POSTING_ID 수:", len(not_found_ids))
print(not_found_ids[:20])

NameError: name 'update_df' is not defined